# Reaching what you can't wait for — steered MD, umbrella sampling & free energies
Notebooks 1–2 ran *unbiased* Trp-cage MD, and §2.6 showed the hard truth: on the timescale a notebook — or even our 10 ns reference — can afford, the interesting events are barely sampled, and the defining cage never opens at all. When you can't wait for a transition, you **bias** the simulation to make it happen and then **remove the bias** to recover the true thermodynamics. That family of tricks is **enhanced sampling**.

There are several families (replica exchange / tempering, metadynamics, adaptive biasing force, …); this notebook demonstrates the most direct one — **biasing a collective variable (CV)**: pick a coordinate that captures the transition and put a harmonic restraint on it. We use that restraint in two ways:
- **Steered MD** (§3.3–3.4): *ramp* the restraint's center outward to drive the cage open — fast, but it distorts the thermodynamics.
- **Umbrella sampling + MBAR** (§3.5): *hold* the restraint at a ladder of fixed centers spanning closed→open, then statistically stitch the windows back together — removing the bias — into a **free-energy profile** along the coordinate, called a **potential of mean force (PMF)**: the effective free energy as a function of the CV, with every *other* degree of freedom averaged out.

**Roadmap:** unbiased MD hits a wall (§3.0–3.1) → a steered pull drives the cage open, judged by *independent* referees (§3.2–3.4) → umbrella windows + MBAR turn the push into a PMF (§3.5). Throughout, the event is the Trp-cage's namesake: cracking Trp6 out of its cage.

In [ ]:
# --- environment on-ramp: make sure the MD stack + the modules are importable in THIS kernel ---
# (identical pattern to fig1/fig2 -- check core deps, provision on Colab, else point at the 'MD tutorial' env)
import importlib.util, sys, os, subprocess, glob
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj", "py3Dmol", "pymbar") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:                          # Colab: provision the stack
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "py3Dmol", "pymbar"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:                                                            # still missing -> almost always the WRONG KERNEL
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
_BASE = os.environ.get("MDTUTORIAL_BASE", "https://raw.githubusercontent.com/OWNER/REPO/main")
for _mod in ("mdtutorial.py", "mdtviz.py", "pull_screen.py"):           # grab the shipped modules if absent
    if not os.path.exists(_mod) and importlib.util.find_spec(_mod[:-3]) is None:
        import urllib.request
        try: urllib.request.urlretrieve(f"{_BASE}/{_mod}", _mod); print("fetched", _mod)
        except Exception as e: print("Place", _mod, "next to this notebook.", e)
import numpy as np, mdtraj as md, matplotlib.pyplot as plt
import openmm, openmm.app as app
import mdtutorial as mdt, mdtviz, pull_screen as steer     # steered-MD/umbrella/MBAR harness + viewers
REF = "reference_10ns"; TOP = os.path.join(REF, "structures", "protein.pdb")

In [ ]:
# --- Knobs: the levers you actually turn. Each is explained; defaults give a fast-but-honest first run. ---
SOLVENT  = "explicit"     # "explicit" = TIP3P water + PME (the physically honest model; used by every run below).
                          # "implicit" = GB continuum, no water: ~10-20x faster -> switch to it when you want to
                          #              pull FARTHER or run LONGER (past ~200 ps) than explicit affords here.
TEMPS    = [300, 315, 330] # pull-screen temperatures to compare (§3.3): 300 K = reference, 315 K ≈ Zhou's melting onset, 330 K hotter.  [dev: dial back to 1–2 before ship]
PULL_PS  = 200            # length of each steered pull, ps (§3.3 CV screen and §3.4 movie). Longer = smoother, slower.  [dev value; ship default ≈100]
NWIN     = 12             # umbrella windows spanning folded→open (§3.5). More windows = better adjacent overlap for MBAR.
WIN_PS   = 20             # sampling ps per umbrella window (§3.5). Longer = tighter PMF error bars.
_avail = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
PLATFORM = next((p for p in ("CUDA", "OpenCL", "CPU") if p in _avail), "CPU")   # CUDA=Colab GPU, OpenCL=Apple GPU, CPU=fallback
print(f"platform {PLATFORM} | {SOLVENT} solvent | temps {TEMPS} K | pull {PULL_PS} ps | umbrella {NWIN}x{WIN_PS} ps")

## 3.0  Motivation: a force on a single mode
Enhanced sampling means **choosing one coordinate and pushing on it.** Here is the coordinate for the cage — the indole→poly-Pro distance — with its free energy **measured from the unbiased reference** (all six seeds, −k_BT·ln P), before we run anything. The data already shows a **folded well** (~4.7 Å) and, past ~6 Å, a **wall**: the unbiased trajectories run out having climbed only ~4–5 k_BT, never reaching the barrier top or the open state, so the free energy beyond is simply *unknown*. The **moving harmonic restraint** ½k(x−r₀)² (red) is the handle we bolt onto this one coordinate — ramping r₀ outward drags the system up the wall (§3.3), and removing the bias afterward is how we'll *try* to recover the barrier's true height (§3.5).

In [ ]:
# Ground the motivation in DATA: free energy along the real cage-opening coordinate (indole->poly-Pro),
# measured from the unbiased reference (all seeds, -kT ln P). No simulation yet -- just what we already know.
_ring = md.load(TOP).topology.select('resSeq 6 and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2')
_lid  = md.load(TOP).topology.select('resSeq 17 18 19 and name CA')
_cvf  = lambda tr: np.linalg.norm(tr.xyz[:, _ring].mean(1) - tr.xyz[:, _lid].mean(1), axis=1) * 10
_cv   = np.concatenate([_cvf(md.load(f, top=TOP)[::5])
                        for f in sorted(glob.glob(os.path.join(REF, 'md_output', 'traj_*.dcd')))])
_kT = 0.00831446261815324 * TEMPS[0]
_h, _e = np.histogram(_cv, bins=30, density=True); _mc = 0.5 * (_e[:-1] + _e[1:]); _cnt = np.histogram(_cv, bins=_e)[0]
_G = -_kT * np.log(_h + 1e-12); _G -= np.nanmin(_G[_cnt >= 5]); _G[_cnt < 20] = np.nan   # keep only well-sampled bins
_wall = np.nanmax(_mc[np.isfinite(_G)])                                                  # where unbiased data runs out
plt.figure(figsize=(7.2, 4.3))
plt.plot(_mc, _G / _kT, 'o-', color='navy', ms=4, label='free energy, measured from unbiased MD')
plt.axvspan(_wall, 9.3, color='0.93')
_xw = np.linspace(_wall - 0.1, 8.2, 40); _gw = np.nanmax(_G[np.isfinite(_G)]) / _kT       # schematic unknown barrier
plt.plot(_xw, _gw + 9 * ((_xw - _wall + 0.1) / (8.2 - _wall + 0.1)) ** 1.7, ':', color='0.55', lw=1.6)
# "here be dragons": scattered question marks in the unknown territory, varied size/orientation
for _qx, _qy, _qs, _qr in [(8.15, 9.7, 15, 0), (8.78, 8.4, 11, 16), (7.55, 9.0, 9, -13),
                           (8.98, 6.6, 13, 9), (8.4, 5.0, 10, -7), (7.72, 7.5, 8, 22)]:
    plt.text(_qx, _qy, '?', fontsize=_qs, color='0.5', ha='center', va='center', rotation=_qr)
plt.text(8.1, 3.5, 'unbiased MD stops at the wall (~%.0f Å);\nthe barrier beyond is unmeasured\n(§3.5 will *try* to reach it)' % _wall,
         ha='center', va='top', fontsize=8, color='0.4')
# a little old-map sea serpent in the blank -- here be dragons
_sx = np.linspace(6.55, 7.5, 140); _sy = 1.7 + 0.26 * np.sin((_sx - 6.55) * 9.5)
plt.plot(_sx, _sy, color='0.55', lw=1.7, solid_capstyle='round')
plt.plot(_sx[-1], _sy[-1], 'o', color='0.55', ms=7)                              # head
plt.plot([_sx[-1] + 0.02], [_sy[-1] + 0.07], '.', color='white', ms=3.5)         # eye
plt.plot([_sx[-1] + 0.02, _sx[-1] + 0.16], [_sy[-1] - 0.04, _sy[-1] - 0.12], color='firebrick', lw=1.0)  # tongue
plt.annotate('to see past the wall, we push the system\ntoward it — out of the well  (§3.3)',
             xy=(6.12, 4.55), xytext=(4.18, 7.8), fontsize=8.5, color='firebrick',
             arrowprops=dict(arrowstyle='->', color='firebrick', lw=1.4, connectionstyle='arc3,rad=-0.2'))
plt.xlim(4, 9.3); plt.ylim(-0.6, 10.5); plt.xlabel('indole→poly-Pro distance (Å)'); plt.ylabel('free energy (k$_B$T)')
plt.title('The one coordinate we push on — grounded in the real data'); plt.legend(fontsize=8, loc='upper left'); plt.show()
print(f'From the unbiased reference alone: folded well at {_mc[np.nanargmin(_G)]:.2f} Å, data runs out by '
      f'{_wall:.1f} Å having climbed only ~{np.nanmax(_G)/_kT:.0f} k_BT — never reaching the barrier or open state.')

## 3.1  The timescale wall — and Zhou's clock
Unbiased MD **did not open the cage**: across the 6-seed 10 ns reference, Trp6's side-chain SASA never exceeds ~67 Å² (a fully exposed Trp is ~200) and Cα-RMSD stays under 3.3 Å — the molecule sits deep in the folded well the whole time. Yet **Zhou (2003)** mapped Trp-cage's folding free-energy landscape and saw folding complete from the intermediate in ~20 ns, with the **Asp9–Arg16 salt bridge** forming late. So the transition *exists* on a reachable timescale — we just don't catch it spontaneously. **Can we run Zhou's folding backwards by pushing?**

In [ ]:
# See the wall, don't just read it. Over a full 10 ns unbiased seed -- ~50x longer than a steered pull --
# how far does Trp6 actually move? Overlay its first and last frame, and quote the displacement + exposure.
_seed = md.load(sorted(glob.glob(os.path.join(REF, 'md_output', 'traj_*.dcd')))[0], top=TOP)
_seed.superpose(_seed, 0)                                          # align out overall tumbling
_sc = _seed.topology.select('resSeq 6 and sidechain'); _trp = _seed.topology.select('resSeq 6')
_sasa = md.shrake_rupley(_seed[::20], mode='atom')[:, _sc].sum(1) * 100
_move = md.rmsd(_seed[-1], _seed[0], atom_indices=_trp)[0] * 10    # Trp6 heavy-atom RMSD, first -> last (Å)
os.makedirs('es_structures', exist_ok=True)
_seed[0].save_pdb('es_structures/ref_first.pdb'); _seed[-1].save_pdb('es_structures/ref_last.pdb')
print(f'Trp6 SASA over the 10 ns seed: max {_sasa.max():.0f} Å²  (a fully exposed Trp is ~200 Å²) -- the cage stays shut.')
print(f'First -> last frame, Trp6 moved just {_move:.1f} Å (heavy-atom RMSD). Over ~50x the length of a steered pull, it only breathes.')
print('   BLUE = first frame,  ORANGE = last frame -- the two Trp6 sticks nearly coincide.')
mdtviz.overlay_view('es_structures/ref_first.pdb', 'es_structures/ref_last.pdb', highlight_resi=6)

## 3.2  Where to push, and how to measure it
**A tutorial choice, stated plainly.** We are *not* deriving the true reaction coordinate for cage opening — that's a hard, separate problem (committor analysis and friends). We screened a handful of simple, restrainable distances against Trp6 exposure (SASA) on the unbiased reference (`cage_cv_screen.py`, plotted below), and two lessons fall out:

- **Correlation isn't enough.** The single best-correlating distance is indole→*cage-centroid* (r≈0.55) — but that centroid is an average over the pocket-lining residues, which spread and reorganize *as the cage opens*. The moment you start prying the cage apart, the reference you're measuring against stops meaning much: a drifting, ill-conditioned coordinate.
- **A CV must stay well-defined across the whole transition.** So we bias **indole→poly-Pro lid** (r≈0.47, a close second): the distance from the **Trp6 indole ring** to the **Pro17–Pro18–Pro19** segment — a rigid poly-proline that caps the indole and stays put whether the cage is shut or open, so the distance to it is a persistent, cleanly-measurable landmark. §3.3 confirms steering it opens the cage *locally* (Trp6 exposes; global RMSD stays low).

It's a reasonable, teachable choice — picked for being well-defined and locally-actionable, not for being *the* coordinate.

**How the force is applied — steered MD:** a harmonic restraint U = ½k(CV − r₀)² whose center r₀ is ramped outward, dragging the CV and peeling the poly-proline lid off the indole. **Independent referees (never biased):** Trp6 **SASA** (cage exposure), the **Asp9–Arg16 salt-bridge** distance, and **Cα-RMSD**. Their job isn't to score a 'success' — it's to watch what pushing this *one* coordinate does to the rest of the molecule, measured by things we didn't push on (never read off only what you biased).

In [ ]:
# WHY indole->polyPro? The DATA: how well does each candidate distance track Trp6 exposure (SASA) across
# the 6-seed unbiased reference (cage_cv_screen.py)? Correlation ranks them; well-definedness picks the winner.
_RING = 'resSeq 6 and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2'
_cen = lambda t, s: t.xyz[:, t.topology.select(s), :].mean(1)
_S, _C = [], {k: [] for k in ['indole→polyPro lid', 'salt bridge (Asp9-Arg16)', 'indole→cage centroid',
                              'indole→Tyr3 ring', 'Trp6 burial depth']}
for _f in sorted(glob.glob(os.path.join(REF, 'md_output', 'traj_*.dcd'))):
    _t = md.load(_f, top=TOP)[::10]
    _S.append(md.shrake_rupley(_t, mode='atom')[:, _t.topology.select('resSeq 6 and sidechain')].sum(1) * 100)
    _r = _cen(_t, _RING)
    _C['indole→polyPro lid'].append(np.linalg.norm(_r - _cen(_t, 'resSeq 17 18 19 and name CA CB CG'), axis=1) * 10)
    _asp = _t.topology.select('resSeq 9 and name OD1 OD2'); _arg = _t.topology.select('resSeq 16 and name NH1 NH2 NE')
    _C['salt bridge (Asp9-Arg16)'].append(md.compute_distances(_t, [(a, b) for a in _asp for b in _arg]).min(1) * 10)
    _C['indole→cage centroid'].append(np.linalg.norm(_r - _cen(_t, 'resSeq 3 11 12 14 18 19 and name CA'), axis=1) * 10)
    _C['indole→Tyr3 ring'].append(np.linalg.norm(_r - _cen(_t, 'resSeq 3 and name CG CD1 CD2 CE1 CE2 CZ'), axis=1) * 10)
    _C['Trp6 burial depth'].append(np.linalg.norm(_cen(_t, 'resSeq 6 and sidechain') - _cen(_t, 'name CA'), axis=1) * 10)
_S = np.concatenate(_S); _C = {k: np.concatenate(v) for k, v in _C.items()}
_ranked = sorted(_C, key=lambda k: -abs(np.corrcoef(_C[k], _S)[0, 1])); _pick = 'indole→polyPro lid'
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), layout='constrained')
_rs = [abs(np.corrcoef(_C[k], _S)[0, 1]) for k in _ranked]
ax[0].barh(range(len(_ranked)), _rs, color=['firebrick' if k == _pick else '0.7' if k == _ranked[0] else 'steelblue' for k in _ranked])
ax[0].set_yticks(range(len(_ranked))); ax[0].set_yticklabels(_ranked, fontsize=8); ax[0].invert_yaxis()
ax[0].set_xlabel('|correlation| with Trp6 SASA'); ax[0].set_title('correlation ranking', fontsize=9); ax[0].set_xlim(0, 0.85)
_yi = {k: i for i, k in enumerate(_ranked)}
ax[0].text(_rs[_yi[_pick]] + 0.01, _yi[_pick], ' ← we bias this\n    (well-defined, local)', va='center', fontsize=7, color='firebrick')
ax[0].text(_rs[0] + 0.01, 0, ' top r, but a\n    drifting reference', va='center', fontsize=7, color='0.4')
for _a, _k in zip(ax[1:], ['indole→polyPro lid', 'salt bridge (Asp9-Arg16)']):
    _a.hist2d(_C[_k], _S, bins=50, cmap='turbo', cmin=1)
    _a.set_xlabel('%s  (Å)' % _k); _a.set_ylabel('Trp6 SASA (Å²)')
    _a.set_title('%s\nr = %+.2f' % (_k.split(' (')[0], np.corrcoef(_C[_k], _S)[0, 1]), fontsize=9)
plt.show()
print('ranked by |r| with SASA: ' + ' | '.join('%s %+.2f' % (k, np.corrcoef(_C[k], _S)[0, 1]) for k in _ranked))
print(f'-> we bias {_pick} (r={np.corrcoef(_C[_pick], _S)[0,1]:+.2f}): not the top correlate, but well-defined, and it opens the cage locally (3.3).')

In [ ]:
pdb = app.PDBFile(TOP); mdtop = md.load(TOP).topology; x0 = md.load(TOP)
sel = steer.selections(mdtop, x0)                 # referee atom selections
CVS = steer.cage_cvs(mdtop, pdb.positions)        # the 5 candidate biasing CVs
ring = mdtop.select('resSeq 6 and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2'); lid = mdtop.select('resSeq 17 18 19 and name CA')
cv_polyPro = lambda tr: np.linalg.norm(tr.xyz[:, ring].mean(1) - tr.xyz[:, lid].mean(1), axis=1) * 10
print('candidate CVs:', list(CVS)); print(f'folded indole→polyPro distance = {cv_polyPro(x0)[0]:.1f} Å')
print('   ORANGE = Trp6 indole  ·  BLUE = Pro17-19 lid  (the biased CV = orange↔blue distance)  ·  RED/GREEN = Asp9/Arg16 salt bridge')
mdtviz.cv_groups_view(TOP, [('6', 'orangeCarbon'), ('17-19', 'blueCarbon'), ('9', 'redCarbon'), ('16', 'greenCarbon')])

## 3.3  Results — the cage opens on one coordinate, not the other
The panels below make the case directly. Each **row** biases one coordinate (left) and watches the **independent** Trp6-SASA referee (right), on a **matched scale** so the 67 Å² unbiased ceiling lines up across both:

- **Bias indole→poly-Pro lid** and SASA climbs with the pull, past the ceiling — the cage opens — while global Cα-RMSD stays low (a *local* opening, not unfolding; see the printout).
- **Bias the salt bridge** and its Asp9–Arg16 distance opens right up, but Trp6 SASA barely moves: the cage stays shut. Breaking the salt bridge is **not sufficient** to open the cage.

So the salt bridge is **coupled** to cage opening (part of the folded latch) but is **not the driver** — the reaction-coordinate lesson: a coordinate can correlate with a transition without being what *causes* it. These are single fast pulls, so read them qualitatively; the converged, quantitative statement is the free-energy profile in §3.5.

In [ ]:
# Steer each CV and watch TWO things over time: the coordinate we push (left) and the INDEPENDENT
# referee, Trp6 SASA (right). Push indole->polyPro and SASA climbs with it; push the salt bridge and its
# distance opens but SASA barely moves -- coupled to cage opening, not the driver. Matched SASA axes.
fig, ax = plt.subplots(2, 2, figsize=(10, 6.6), sharex=True, layout='constrained')
_lss = ['-', '--', ':', '-.']; _sasa_all = []
for _i, (_cv, _note) in enumerate([('indole→polyPro lid', 'opens the cage'), ('salt bridge', 'does not open the cage')]):
    _fac, _delta = CVS[_cv]
    for _j, _T in enumerate(TEMPS):
        _sa, _sb, _rmsd, _traj = steer.run_pull(pdb, mdtop, _fac, _delta, _T, PULL_PS, PLATFORM, sel, solvent=SOLVENT)
        _t = np.linspace(0, PULL_PS, len(_sa))
        _biased = cv_polyPro(_traj) if _cv == 'indole→polyPro lid' else _sb        # the coordinate we actually pushed
        _ls = _lss[_j % 4]
        ax[_i, 0].plot(_t, _biased, _ls, label=f'{_T} K'); ax[_i, 1].plot(_t, _sa, _ls, label=f'{_T} K')
        _sasa_all.append(_sa)
        print(f'{_cv:20s} {_T}K: pushed {_biased[0]:.1f}->{_biased.max():.1f} Å | Trp6 SASA {_sa[0]:.0f}->{_sa.max():.0f} Å² | RMSD ->{_rmsd[-1]:.1f} Å')
    ax[_i, 0].set_ylabel('%s\n(Å, the coordinate pushed)' % _cv, fontsize=8); ax[_i, 1].set_ylabel('Trp6 SASA (Å²)')
    ax[_i, 1].axhline(67, color='0.6', ls=':', lw=1)                                # unbiased 10 ns exposure ceiling
    ax[_i, 0].text(0.03, 0.93, 'bias %s → %s' % (_cv, _note), transform=ax[_i, 0].transAxes, fontsize=9, va='top',
                   color=('firebrick' if _i == 0 else 'steelblue'))
_ymax = max(s.max() for s in _sasa_all) * 1.05
for _a in ax[:, 1]:
    _a.set_ylim(0, _ymax)                                                           # matched SASA axis -> honest comparison
ax[0, 0].set_title('the coordinate we push', fontsize=9); ax[0, 1].set_title('Trp6 SASA — independent referee', fontsize=9)
ax[1, 0].set_xlabel('time (ps)'); ax[1, 1].set_xlabel('time (ps)'); ax[0, 1].legend(fontsize=7, title='temperature')
plt.show()

## 3.4  What it looks like on the molecule
A **controlled comparison**, not a loaded one: both trajectories are the **same length**, from the **same folded start**, in the **same explicit solvent** — the *only* difference is that the right one has the steering force applied. **Left (unbiased):** the cage breathes but stays folded. (The quantitative claim that unbiased MD *never* opens it is §3.1's 10 ns reference — this short matched clip just shows the folded motion under identical conditions, so the two panels differ by the force alone, not by how long we watched.) **Right (steered `indole→polyPro`):** as the restraint center ramps outward the force works the poly-Pro lid loose and Trp6 shifts outward — but be honest about the magnitude: it's a **subtle outward tilt of the indole, not a dramatic ejection**. Depending on the run it may not travel far, and Trp6 **never becomes fully solvent-exposed** (recall §3.1 — even the most 'open' frames sit far below a bare Trp's ~200 Å²). What you're watching is the *onset* of cage opening, completing late in the pull as the force finally wins — not a spontaneous event. Trp6 is orange sticks; the faint grey ghost is its folded starting position. *This* is the motion we've been calling 'cage opening.'

In [ ]:
# Two aligned trajectories (subsampled): unbiased (stays shut) vs steered (opens), animated side by side.
_shut = steer.run_unbiased(pdb, mdtop, TEMPS[0], PULL_PS, PLATFORM, save_ps=2.0, solvent=SOLVENT)  # same length as the pull -> equal-duration side-by-side
_fac, _delta = CVS['indole→polyPro lid']
*_, _open = steer.run_pull(pdb, mdtop, _fac, _delta, TEMPS[0], PULL_PS, PLATFORM, sel, save_ps=2.0, solvent=SOLVENT)
steer.save_aligned(_shut, x0, 'es_structures/traj_shut.pdb')      # superpose -> shared folded orientation
steer.save_aligned(_open, x0, 'es_structures/traj_open.pdb')
print('   LEFT: unbiased — cage stays shut            RIGHT: steered — cage opens (watch Trp6 swing out)')
mdtviz.dual_view('es_structures/traj_shut.pdb', 'es_structures/traj_open.pdb', highlight_resi=6)

## 3.5  From a push to a free energy — umbrella sampling + MBAR
This is the most abstract section, so we'll be explicit — nobody absorbs MBAR on the first read.

**Why the pull's free energy is wrong.** A fast steered pull is a *non-equilibrium* process: we yank the restraint faster than the surroundings can relax, so some of the work we did went into friction and dissipation, not into the free energy. (This is Jarzynski's territory — the *average* pull work overestimates ΔF; only a careful reweighting recovers the equilibrium value.) So we can drive the transition with a pull, but we can't read ΔF off it.

**Umbrella sampling — tile the path instead of racing across it.** Rather than *ramping* the restraint, we **park** it at a ladder of fixed centers r₀ spanning closed→open (`centers`) and let each window equilibrate and sample. Each restraint pins the system in a thin slice of the coordinate it would otherwise never visit, so together the windows **tile the whole path, barrier included**. Each window's samples are *biased* (tugged toward its center) — not the equilibrium distribution on their own.

**MBAR — divide the bias back out.** MBAR (multistate Bennett acceptance ratio) is the reweighting that removes it: given every window's samples and the known restraint energy of each, it solves self-consistently for the per-window free energies and assigns each sample an **unbiased weight**; histogramming those weights gives the **PMF**. It's the statistically-optimal, binless successor to WHAM. We use **pymbar's** implementation, which also returns **analytical uncertainties** (the error bars), propagated from the sample statistics — no bootstrap needed.

**Do the windows actually connect?** MBAR can only stitch windows that **overlap** — adjacent windows must sample some common range of the coordinate, or there's no bridge between them. pymbar's **overlap matrix** measures exactly that; we print the smallest adjacent overlap, and a value comfortably above ~0 means the chain from closed to open is unbroken. (Near zero → add windows, spaced closer.)

**Reading the profile.** The PMF **rises** out of the folded well and then **plateaus** — the plateau height (in k_BT) is the free-energy **cost** of prying the cage open. And we **cross-check** it: in the closed well, where the unbiased reference actually sampled, the PMF must agree with the −k_BT·ln P(CV) measured directly from those trajectories (§3.0's curve). Where they overlap they should lie on top of each other — the proof MBAR didn't invent anything — and the **open arm**, past where the reference dies, is the payoff: the free energy of a state unbiased MD never reached.

*Honesty:* this is a deliberately small, fast, illustrative run. The **shape** is robust; the absolute barrier height is **not converged** — treat the number as illustrative and converge it with more/longer windows for anything quantitative.

**Why this works (further reading):** umbrella sampling — Torrie & Valleau, *J. Comput. Phys.* **23**, 187 (1977), doi:10.1016/0021-9991(77)90121-8; MBAR — Shirts & Chodera, *J. Chem. Phys.* **129**, 124105 (2008), doi:10.1063/1.2978177; the non-equilibrium-work view of why a fast pull distorts ΔF — Jarzynski, *Phys. Rev. Lett.* **78**, 2690 (1997), doi:10.1103/PhysRevLett.78.2690.

In [ ]:
fac, delta = CVS['indole→polyPro lid']; K = 4000.0; T = TEMPS[0]
folded = float(np.linalg.norm(x0.xyz[0, ring].mean(0) - x0.xyz[0, lid].mean(0)))   # nm
centers = np.linspace(folded, folded + delta, NWIN)
windows = steer.run_umbrella(pdb, fac, centers, T, PLATFORM, k=K, equil_ps=6, sample_ps=WIN_PS, solvent=SOLVENT)
xA, pmf, err = steer.mbar_pmf_pymbar(windows, centers, K, T)          # pymbar FES: PMF + analytical error (kJ/mol)
_, min_ov = steer.mbar_overlap(windows, centers, K, T)                 # pymbar overlap diagnostic
print(f'windows connected? min adjacent overlap = {min_ov:.3f}  ({"yes" if min_ov > 0.03 else "add windows"})')

# cross-check the CLOSED well against -kT ln P(CV) from the unbiased reference
kT = 0.00831446261815324 * T
cv_u = np.concatenate([cv_polyPro(md.load(f, top=TOP)[::5])                       # pool ALL reference seeds
                       for f in sorted(glob.glob(os.path.join(REF, 'md_output', 'traj_*.dcd')))])
h, e = np.histogram(cv_u, bins=25, density=True); mc = 0.5 * (e[:-1] + e[1:]); pmf_u = -kT * np.log(h + 1e-12)
pmf_u[np.histogram(cv_u, bins=e)[0] < 5] = np.nan   # show only where the unbiased ref truly sampled; the tail is the -kT ln(~0) log-floor, i.e. the state unbiased MD never reaches -- the very reason we umbrella-sample
ov = (mc > xA.min()) & (mc < xA.min() + 2.0) & np.isfinite(pmf_u)      # align on the SAMPLED overlap only (skip masked tail)
if ov.any(): pmf_u += np.interp(mc[ov], xA, pmf).mean() - pmf_u[ov].mean()

plt.figure(figsize=(6.6, 4.3))
plt.errorbar(xA, pmf, yerr=err, fmt='-o', ms=3, color='navy', label='MBAR PMF (umbrella)')
plt.plot(mc, pmf_u, '--', color='firebrick', label='−k$_B$T ln P (unbiased ref, closed well)')
plt.xlabel('indole→polyPro distance (Å)'); plt.ylabel('free energy (kJ/mol)')
plt.title('Cage-opening free-energy profile'); plt.legend(); plt.ylim(bottom=-2); plt.show()
print(f'cage-opening cost ≈ {np.nanmax(pmf):.0f} kJ/mol = {np.nanmax(pmf)/kT:.1f} k_BT '
      f'(illustrative: short {NWIN}-window {SOLVENT}-solvent run; converge with more/longer windows)')